# SIS and ISIS: Screening for Ultra-High Dimensional Data

## Learning Objectives

By completing this notebook, you will:
1. Implement Sure Independence Screening (SIS) and Iterative SIS (ISIS) from scratch
2. Generate a realistic high-dimensional scenario ($p = 5{,}000$, $n = 200$) with known active features
3. Measure how SIS reduces the problem before Lasso without losing active features
4. Compare SIS → Lasso vs direct Lasso on the full feature set (speed and accuracy)
5. Identify when marginal screening fails and why ISIS recovers suppressor variables

## Prerequisites
- Lasso regularisation (Module 4)
- Marginal correlation basics (Module 1)
- Guide 01 of this module: *Sure Independence Screening*

## Estimated Time: 15 minutes

## Setup

We use standard scientific Python. The only non-standard library is `scikit-learn` for Lasso and cross-validation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
from sklearn.linear_model import LassoCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

## 1. Generating a Realistic High-Dimensional Scenario

Real high-dimensional data has block-correlated features — not independent columns. We generate a dataset with:
- $n = 200$ observations, $p = 5{,}000$ features
- 10 correlated feature blocks of 500 features each
- 8 active features (true signal) drawn from different blocks
- 2 suppressor variable pairs (for the ISIS demonstration later)

This mimics genomics or financial signal data where features cluster by pathway or sector.

In [ ]:
def generate_high_dim_data(
    n: int = 200,
    p: int = 5000,
    n_blocks: int = 10,
    n_active: int = 8,
    block_corr: float = 0.7,
    snr: float = 3.0,
    seed: int = 42
):
    """
    Generate a high-dimensional regression dataset with block-correlated features.

    Parameters
    ----------
    n : int
        Number of observations.
    p : int
        Total number of features.
    n_blocks : int
        Number of correlated feature blocks (p must be divisible by n_blocks).
    n_active : int
        Number of features with non-zero coefficients.
    block_corr : float
        Within-block correlation coefficient (equicorrelation structure).
    snr : float
        Signal-to-noise ratio: Var(signal) / Var(noise).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    X : ndarray (n, p)
        Feature matrix, standardised columns.
    y : ndarray (n,)
        Continuous outcome.
    true_active : ndarray
        Indices of truly active features (ground truth).
    beta_true : ndarray (p,)
        True coefficient vector.
    """
    rng = np.random.default_rng(seed)
    block_size = p // n_blocks

    # Build block-correlated design matrix
    # Each block has equicorrelation structure: Cor(X_i, X_j) = block_corr for i != j in same block
    X_blocks = []
    for b in range(n_blocks):
        # Cholesky decomposition of equicorrelation matrix
        # Sigma = (1-rho)*I + rho*11^T, Cholesky: L = sqrt(1-rho)*I + (sqrt(rho) - ...)...
        # Simpler: generate z ~ N(0,I) and u ~ N(0,1), set X = sqrt(rho)*u + sqrt(1-rho)*z
        z = rng.standard_normal((n, block_size))  # idiosyncratic variation
        u = rng.standard_normal(n)                # common factor within block
        block = np.sqrt(block_corr) * u[:, None] + np.sqrt(1 - block_corr) * z
        X_blocks.append(block)

    X = np.hstack(X_blocks)  # (n, p)

    # Place active features: one near the start of each of the first n_active blocks
    # (at position 0 within each block — guaranteed to have marginal signal)
    true_active = np.array([b * block_size for b in range(n_active)])

    # Coefficients: varying magnitudes to simulate heterogeneous signal strength
    beta_true = np.zeros(p)
    beta_magnitudes = np.array([2.0, 1.5, 1.5, 1.0, 1.0, 0.8, 0.8, 0.6])
    signs = rng.choice([-1, 1], size=n_active)
    beta_true[true_active] = beta_magnitudes * signs

    # Generate outcome with specified SNR
    signal = X @ beta_true
    noise_var = np.var(signal) / snr
    y = signal + rng.standard_normal(n) * np.sqrt(noise_var)

    # Standardise X (SIS uses marginal correlations — scale-sensitive)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y, true_active, beta_true


X, y, true_active, beta_true = generate_high_dim_data()
n, p = X.shape
print(f"Dataset shape: n={n}, p={p}")
print(f"True active features: {true_active}")
print(f"True coefficients: {beta_true[true_active].round(2)}")

## 2. Implementing SIS from Scratch

SIS computes the absolute marginal correlation between each feature $X_j$ and $y$, then retains the top $d_n$ features.

The marginal correlation score for standardised features and centred $y$ is:
$$\hat{\omega}_j = \frac{1}{n}|\mathbf{x}_j^\top \mathbf{y}|$$

The canonical threshold is $d_n = \lfloor n / \log n \rfloor$ (Fan & Lv, 2008).

In [ ]:
def sis(
    X: np.ndarray,
    y: np.ndarray,
    d_n: int = None
) -> tuple[np.ndarray, np.ndarray]:
    """
    Sure Independence Screening (Fan & Lv, 2008).

    Computes marginal correlation scores and returns the top-d_n feature indices.

    Parameters
    ----------
    X : ndarray (n, p)
        Feature matrix. Should be column-standardised.
    y : ndarray (n,)
        Outcome vector. Should be centred.
    d_n : int, optional
        Number of features to retain. Defaults to floor(n / log(n)).

    Returns
    -------
    screened_idx : ndarray (d_n,)
        Indices of the top-d_n features, sorted by score descending.
    omega : ndarray (p,)
        Marginal correlation scores for all features.
    """
    n, p = X.shape
    if d_n is None:
        d_n = int(n / np.log(n))  # Fan & Lv canonical threshold

    # Centre y to ensure marginal correlation == inner product / n
    y_ctr = y - y.mean()

    # Marginal correlation scores: |X^T y| / n
    # X is already standardised so this equals |Cor(x_j, y)| (up to y's norm)
    omega = np.abs(X.T @ y_ctr) / n  # shape (p,)

    # Top-d_n features by score
    screened_idx = np.argsort(omega)[::-1][:d_n]

    return screened_idx, omega


# Run SIS
d_n = int(n / np.log(n))
print(f"SIS threshold: d_n = floor({n} / log({n})) = {d_n}")

screened_idx, omega = sis(X, y, d_n=d_n)

# Check the sure screening property: are all true active features in the screened set?
active_recovered = set(true_active) & set(screened_idx)
active_missed = set(true_active) - set(screened_idx)

print(f"\nScreened {len(screened_idx)} features from {p}")
print(f"True active features recovered: {len(active_recovered)}/{len(true_active)}")
if active_missed:
    print(f"Missed (false negatives): {sorted(active_missed)}")
else:
    print("Sure screening property holds: all active features retained")

## 3. Visualising the Screening Step

The histogram of marginal correlation scores shows the distribution of signal. True active features should appear as outliers on the right tail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of all omega scores
ax = axes[0]
ax.hist(omega, bins=100, color='steelblue', alpha=0.7, label='Noise features')

# Highlight true active features
ax.scatter(
    omega[true_active],
    np.zeros(len(true_active)) + 5,
    color='red', s=80, zorder=5, label='True active features'
)

# Screening threshold
threshold_value = np.sort(omega)[::-1][d_n - 1]
ax.axvline(threshold_value, color='orange', linestyle='--', linewidth=2,
           label=f'SIS threshold (d_n={d_n})')

ax.set_xlabel('Marginal correlation score |ω_j|')
ax.set_ylabel('Count')
ax.set_title(f'SIS Score Distribution (p={p})')
ax.legend()

# Right: zoom on top-100 scores
ax = axes[1]
top100_idx = np.argsort(omega)[::-1][:100]
top100_scores = omega[top100_idx]
colors = ['red' if idx in true_active else 'steelblue' for idx in top100_idx]

ax.bar(range(100), top100_scores, color=colors, alpha=0.8)
ax.axvline(d_n - 0.5, color='orange', linestyle='--', linewidth=2,
           label=f'SIS cut-off at rank {d_n}')
ax.set_xlabel('Feature rank')
ax.set_ylabel('Score |ω_j|')
ax.set_title('Top 100 Features (red = true active)')
ax.legend()

# Custom legend patch
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='True active'),
    Patch(facecolor='steelblue', label='Noise'),
]
ax.legend(handles=legend_elements)

plt.tight_layout()
plt.suptitle('Sure Independence Screening — Score Distribution', y=1.02, fontsize=13)
plt.show()

print(f"\nRanks of true active features among all {p} features:")
for feat in sorted(true_active):
    rank = np.sum(omega >= omega[feat])
    print(f"  Feature {feat}: score={omega[feat]:.4f}, rank={rank}")

## 4. SIS → Lasso Pipeline

After screening to $d_n$ features, we run Lasso on the reduced feature set. This is both faster and more numerically stable than running Lasso on all $p$ features.

In [ ]:
def sis_lasso_pipeline(
    X: np.ndarray,
    y: np.ndarray,
    d_n: int = None,
    cv: int = 5
) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Two-step pipeline: SIS screening followed by Lasso selection.

    Parameters
    ----------
    X : ndarray (n, p)
    y : ndarray (n,)
    d_n : int
        SIS screen size. Defaults to floor(n / log(n)).
    cv : int
        Cross-validation folds for Lasso tuning.

    Returns
    -------
    selected_features : ndarray
        Indices of selected features in original feature space.
    coef : ndarray
        Non-zero Lasso coefficients for selected features.
    elapsed : float
        Wall-clock time in seconds.
    """
    t0 = time.perf_counter()

    # Step 1: SIS screening
    screened_idx, _ = sis(X, y, d_n=d_n)
    X_reduced = X[:, screened_idx]

    # Step 2: Lasso on reduced feature set
    lasso = LassoCV(cv=cv, max_iter=10000, n_jobs=-1)
    lasso.fit(X_reduced, y)

    # Map back to original feature indices
    active_in_reduced = np.where(np.abs(lasso.coef_) > 1e-8)[0]
    selected_features = screened_idx[active_in_reduced]
    coef = lasso.coef_[active_in_reduced]

    elapsed = time.perf_counter() - t0
    return selected_features, coef, elapsed


# Run SIS → Lasso
sel_sis_lasso, coef_sis_lasso, time_sis_lasso = sis_lasso_pipeline(X, y)

print("=== SIS → Lasso Pipeline ===")
print(f"Features selected: {len(sel_sis_lasso)}")
print(f"Time: {time_sis_lasso:.2f}s")
print(f"Selected features: {sorted(sel_sis_lasso)}")

# Check true positive / false positive rate
tp = len(set(sel_sis_lasso) & set(true_active))
fp = len(set(sel_sis_lasso) - set(true_active))
fn = len(set(true_active) - set(sel_sis_lasso))
print(f"\nTrue positives: {tp}/{len(true_active)}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")

## 5. Direct Lasso on Full Feature Set

For comparison, run Lasso on all $p = 5{,}000$ features. This demonstrates the computational cost difference and whether the full Lasso recovers more or fewer true features.

In [ ]:
print("Running direct Lasso on all p=5000 features...")
t0 = time.perf_counter()

# Direct Lasso — uses all p features
# Use fewer CV folds to keep runtime manageable
lasso_direct = LassoCV(cv=5, max_iter=5000, n_jobs=-1)
lasso_direct.fit(X, y)

time_direct = time.perf_counter() - t0
sel_direct = np.where(np.abs(lasso_direct.coef_) > 1e-8)[0]

tp_direct = len(set(sel_direct) & set(true_active))
fp_direct = len(set(sel_direct) - set(true_active))
fn_direct = len(set(true_active) - set(sel_direct))

print(f"\n=== Direct Lasso (p=5000) ===")
print(f"Features selected: {len(sel_direct)}")
print(f"Time: {time_direct:.2f}s")
print(f"True positives: {tp_direct}/{len(true_active)}")
print(f"False positives: {fp_direct}")
print(f"False negatives: {fn_direct}")

# Summary comparison
print("\n" + "="*50)
print("COMPARISON SUMMARY")
print("="*50)
print(f"{'Method':<25} {'Time':>8} {'TP':>5} {'FP':>5} {'FN':>5}")
print("-"*50)
print(f"{'SIS → Lasso':<25} {time_sis_lasso:>7.2f}s {tp:>5} {fp:>5} {fn:>5}")
print(f"{'Direct Lasso':<25} {time_direct:>7.2f}s {tp_direct:>5} {fp_direct:>5} {fn_direct:>5}")
print(f"\nSpeedup: {time_direct / time_sis_lasso:.1f}×")

## 6. When Marginal Screening Fails: Suppressor Variables

A *suppressor variable* is a feature that is relevant **conditionally** but has near-zero **marginal** correlation with $y$.

**Setup:** We construct a dataset where $X_1$ and $X_2$ are jointly relevant but $\text{Cor}(X_1, y) \approx 0$.

The model is: $y = \beta_1 X_1 + \beta_2 X_2 + \varepsilon$

With $X_1 \approx -X_2 + \text{noise}$ (highly negatively correlated), the marginal effects of $X_1$ and $X_2$ on $y$ cancel — SIS will miss the suppressor.

In [ ]:
def generate_suppressor_scenario(
    n: int = 200,
    p_noise: int = 498,
    beta1: float = 2.0,
    beta2: float = 2.0,
    suppressor_corr: float = -0.95,
    seed: int = 0
):
    """
    Generate data with a suppressor variable structure.

    Feature 0 (X1): has non-zero beta but near-zero marginal correlation with y.
    Feature 1 (X2): has strong marginal correlation with y.
    Features 2..p-1: pure noise.

    Returns X (n, p), y (n,), true_active ([0, 1])
    """
    rng = np.random.default_rng(seed)
    p = 2 + p_noise

    # X2 is a standard normal variable
    x2 = rng.standard_normal(n)

    # X1 = suppressor_corr * X2 + small_noise
    # This makes X1 highly correlated with X2 but in the opposite direction
    x1 = suppressor_corr * x2 + np.sqrt(1 - suppressor_corr**2) * rng.standard_normal(n)

    # Noise features
    X_noise = rng.standard_normal((n, p_noise))

    X = np.column_stack([x1, x2, X_noise])

    # True model: both x1 and x2 contribute equally
    # Since x1 ≈ -x2, their marginal effects on y cancel:
    # Cor(x1, y) ≈ beta1 + beta2 * suppressor_corr ≈ 2 + 2*(-0.95) = 0.1 ≈ 0
    signal = beta1 * x1 + beta2 * x2
    noise_std = np.std(signal) / 3.0  # SNR = 3
    y = signal + rng.standard_normal(n) * noise_std

    # Standardise
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    true_marginal_corr_x1 = np.corrcoef(X[:, 0], y)[0, 1]
    true_marginal_corr_x2 = np.corrcoef(X[:, 1], y)[0, 1]
    print(f"Marginal Cor(X1, y) = {true_marginal_corr_x1:.4f}  ← suppressor: near zero")
    print(f"Marginal Cor(X2, y) = {true_marginal_corr_x2:.4f}  ← strong signal")
    print(f"True active features: [0, 1]")

    return X, y, np.array([0, 1])


X_sup, y_sup, true_active_sup = generate_suppressor_scenario()

### 6.1 SIS Fails on the Suppressor

Because $X_1$ has near-zero marginal correlation with $y$, SIS assigns it a low score and discards it.

In [ ]:
n_sup = X_sup.shape[0]
d_n_sup = int(n_sup / np.log(n_sup))

screened_sup, omega_sup = sis(X_sup, y_sup, d_n=d_n_sup)

print(f"SIS screened set (top {d_n_sup} features)")
print(f"Feature 0 (suppressor) in screened set? {0 in screened_sup}")
print(f"Feature 1 (strong signal) in screened set? {1 in screened_sup}")
print(f"\nOmega scores: X0={omega_sup[0]:.4f}, X1={omega_sup[1]:.4f}")
rank_x0 = np.sum(omega_sup >= omega_sup[0])
rank_x1 = np.sum(omega_sup >= omega_sup[1])
print(f"Ranks: X0 is rank {rank_x0} of {len(omega_sup)}, X1 is rank {rank_x1}")
print(f"\nSIS FAILS: Feature 0 (suppressor) has rank {rank_x0}, far below threshold d_n={d_n_sup}")

## 7. Implementing ISIS: Recovering the Suppressor

ISIS residualises $y$ against already-selected features in each iteration. After selecting $X_2$, the residual captures the unexplained variation driven by $X_1$ — making $X_1$'s marginal relevance visible.

In [ ]:
def isis(
    X: np.ndarray,
    y: np.ndarray,
    d_n: int = None,
    max_iter: int = 5,
    lasso_cv: int = 3
) -> tuple[np.ndarray, list]:
    """
    Iterative SIS (ISIS) — Fan & Lv (2008).

    At each iteration:
    1. Compute residuals from the current model
    2. Screen unselected features against residuals
    3. Add top features to the selected set
    4. Apply Lasso to prune the expanded set
    5. Repeat until convergence

    Parameters
    ----------
    X : ndarray (n, p)
    y : ndarray (n,)
    d_n : int
        Number of features added per iteration. Defaults to floor(n / log(n)).
    max_iter : int
        Maximum number of ISIS iterations.
    lasso_cv : int
        CV folds for within-iteration Lasso pruning.

    Returns
    -------
    selected : ndarray
        Final selected feature indices.
    history : list of sets
        Selected set at each iteration (for diagnostics).
    """
    n, p = X.shape
    if d_n is None:
        d_n = int(n / np.log(n))

    selected = set()       # currently selected feature indices
    history = [set()]

    for t in range(max_iter):
        remaining = np.array([j for j in range(p) if j not in selected])

        if len(selected) == 0:
            # First iteration: standard SIS
            residuals = y - y.mean()
        else:
            # Compute OLS residuals from current selected set
            X_sel = X[:, sorted(selected)]
            # Use Lasso rather than OLS to avoid overfitting when |selected| grows
            lasso_cur = LassoCV(cv=lasso_cv, max_iter=5000)
            lasso_cur.fit(X_sel, y)
            residuals = y - lasso_cur.predict(X_sel)

        # Screen remaining features against residuals
        residuals_ctr = residuals - residuals.mean()
        omega_remaining = np.abs(X[:, remaining].T @ residuals_ctr) / n

        # Add top d_n from remaining
        n_add = min(d_n, len(remaining))
        top_in_remaining = np.argsort(omega_remaining)[::-1][:n_add]
        new_features = set(remaining[top_in_remaining])
        candidate_set = selected | new_features

        # Prune with Lasso
        X_cand = X[:, sorted(candidate_set)]
        lasso_prune = LassoCV(cv=lasso_cv, max_iter=5000)
        lasso_prune.fit(X_cand, y)
        cand_list = sorted(candidate_set)
        pruned = {cand_list[j] for j in np.where(np.abs(lasso_prune.coef_) > 1e-8)[0]}

        history.append(pruned.copy())

        # Check convergence
        if pruned == selected:
            print(f"ISIS converged at iteration {t+1}")
            break

        selected = pruned

    return np.array(sorted(selected)), history


# Run ISIS on the suppressor scenario
print("Running ISIS on suppressor scenario...")
selected_isis, history_isis = isis(X_sup, y_sup, max_iter=5)

print(f"\nISIS selected features: {sorted(selected_isis)}")
print(f"Feature 0 (suppressor) recovered: {0 in selected_isis}")
print(f"Feature 1 (strong signal) recovered: {1 in selected_isis}")

print("\nIteration history:")
for t, sel in enumerate(history_isis):
    print(f"  Iteration {t}: selected = {sorted(sel)}")

## 8. Visualising Why ISIS Works

After iteration 1 selects $X_2$, the residual $r_1 = y - \hat{y}_{\{X_2\}}$ contains the unexplained variation driven by $X_1$. We can see this by plotting the marginal correlations at each ISIS iteration.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Compute scores at each residualisation step manually for visualisation
y_sup_ctr = y_sup - y_sup.mean()

# Step 0: raw y (first SIS pass)
omega_0 = np.abs(X_sup.T @ y_sup_ctr) / n_sup

# Step 1: residual after removing X2 (feature 1)
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(X_sup[:, [1]], y_sup)
resid_1 = y_sup - lr.predict(X_sup[:, [1]])
resid_1_ctr = resid_1 - resid_1.mean()
omega_1 = np.abs(X_sup.T @ resid_1_ctr) / n_sup

for ax, omega, title, step in zip(
    axes,
    [omega_0, omega_1, omega_0],
    ['Step 0: Raw y (initial SIS)', 'Step 1: Residual after X2', 'SIS vs ISIS comparison'],
    [0, 1, None]
):
    if step is not None:
        # Show score distribution
        ax.hist(omega[2:], bins=50, color='steelblue', alpha=0.6, label='Noise features')
        ax.axvline(omega[0], color='red', linewidth=2.5, linestyle='-', label='X0 (suppressor)')
        ax.axvline(omega[1], color='green', linewidth=2.5, linestyle='-', label='X1 (strong)')
        threshold_here = np.sort(omega)[::-1][d_n_sup - 1]
        ax.axvline(threshold_here, color='orange', linestyle='--', label=f'Threshold d_n={d_n_sup}')
        ax.set_xlabel('Score |ω_j|')
        ax.set_ylabel('Count')
        ax.set_title(title)
        ax.legend(fontsize=9)

# Panel 3: direct comparison of X0 score across steps
ax = axes[2]
steps = ['Raw y (SIS)', 'After removing X2 (ISIS step 2)']
x0_scores = [omega_0[0], omega_1[0]]
x1_scores = [omega_0[1], omega_1[1]]
x_pos = np.arange(2)
width = 0.35

bars1 = ax.bar(x_pos - width/2, x0_scores, width, color='red', alpha=0.8, label='X0 (suppressor)')
bars2 = ax.bar(x_pos + width/2, x1_scores, width, color='green', alpha=0.8, label='X1 (strong)')

# Add threshold line for step 0 and step 1
thresh0 = np.sort(omega_0)[::-1][d_n_sup - 1]
thresh1 = np.sort(omega_1)[::-1][d_n_sup - 1]
ax.hlines(thresh0, -0.5, 0.5, colors='orange', linestyles='--', label=f'Threshold step 0')
ax.hlines(thresh1, 0.5, 1.5, colors='purple', linestyles='--', label=f'Threshold step 1')

ax.set_xticks(x_pos)
ax.set_xticklabels(steps, fontsize=10)
ax.set_ylabel('Marginal correlation score')
ax.set_title('ISIS Unmasks the Suppressor')
ax.legend(fontsize=9)

# Annotation
ax.annotate('Below threshold\n→ SIS discards X0',
            xy=(0 - width/2, omega_0[0]), xytext=(0.3, omega_0[0] - 0.02),
            fontsize=8, color='red')
ax.annotate('Above threshold\n→ ISIS selects X0',
            xy=(1 - width/2, omega_1[0]), xytext=(0.6, omega_1[0] + 0.01),
            fontsize=8, color='darkred')

plt.tight_layout()
plt.suptitle('Why ISIS Recovers Suppressor Variables', y=1.02, fontsize=13)
plt.show()

## 9. Simulation Study: Sure Screening Property Empirically

We verify the sure screening property across 100 Monte Carlo runs: how often does SIS include all true active features in the screened set?

In [ ]:
def run_screening_simulation(
    n_reps: int = 100,
    n: int = 200,
    p: int = 2000,
    n_active: int = 8,
    snr: float = 3.0
) -> dict:
    """Monte Carlo study of the sure screening property."""
    results = {'sis_recall': [], 'sis_selected_count': []}

    for rep in range(n_reps):
        X_r, y_r, active_r, _ = generate_high_dim_data(
            n=n, p=p, n_active=n_active, snr=snr, seed=rep
        )
        d_n_r = int(n / np.log(n))
        screened_r, _ = sis(X_r, y_r, d_n=d_n_r)

        # Recall: fraction of true active features recovered
        recall = len(set(active_r) & set(screened_r)) / len(active_r)
        results['sis_recall'].append(recall)
        results['sis_selected_count'].append(len(screened_r))

    return results


print("Running Monte Carlo simulation (100 reps)...")
sim_results = run_screening_simulation(n_reps=100)

recall_arr = np.array(sim_results['sis_recall'])
print(f"\nSure Screening Property (p=2000, n=200, SNR=3):")
print(f"  Full recall (all active found): {np.mean(recall_arr == 1.0):.1%}")
print(f"  Mean recall: {recall_arr.mean():.3f}")
print(f"  Min recall: {recall_arr.min():.3f}")
print(f"  Mean features screened: {np.mean(sim_results['sis_selected_count']):.1f} / {int(200/np.log(200))}")

# Plot recall distribution
fig, ax = plt.subplots(figsize=(8, 4))
unique_recall, counts = np.unique(recall_arr, return_counts=True)
ax.bar([f"{r:.2f}" for r in unique_recall], counts / len(recall_arr),
       color='steelblue', alpha=0.8)
ax.set_xlabel('SIS Recall of True Active Features')
ax.set_ylabel('Frequency')
ax.set_title(f'Sure Screening Property: {100*np.mean(recall_arr==1.0):.0f}% full recall over 100 reps')
plt.tight_layout()
plt.show()

## Summary

### Key Takeaways

1. **SIS is fast and theoretically grounded**: The marginal correlation score $\hat{\omega}_j$ reduces $p = 5{,}000$ to $d_n \sim 38$ features in linear time, with the sure screening property guaranteeing that true active features survive.

2. **SIS → Lasso dominates direct Lasso**: The two-step pipeline is faster and achieves comparable or better selection accuracy by stabilising the Lasso computation.

3. **Suppressor variables break SIS**: Features with near-zero marginal correlation are discarded even when conditionally relevant. ISIS recovers them by screening against residuals.

4. **ISIS recovers suppressors in 2–3 iterations**: After selecting the strongly marginal feature in step 1, the residual correlation of the suppressor becomes non-zero — ISIS detects it in step 2.

### What's Next

Notebook 02 (`02_sparse_pca_selection.ipynb`) covers structured sparsity: Sparse PCA for discovering feature groups and Group Lasso for group-level selection.

### Additional Resources

- **Fan & Lv (2008)**: The original SIS paper — JRSS-B 70(5), 849–911
- **Guide 01** in this module: full theoretical derivation of the sure screening property
- `sklearn.linear_model.LassoCV`: implementation details for the Lasso step